1. Imports & Configuration

In [2]:
import os
import time
import hashlib
from pathlib import Path

import ollama
import chromadb
from chromadb.config import Settings

In [ ]:

CONFIG = {
    # Models
    "llm_model":       "gemma2:2b",
    "embed_model":     "nomic-embed-text",

    # Paths
    "documents_dir":   r"C:\1 Raji Sivani\UTD\Sem 4\Agentic AI\Project 1\Lion_kings_RAG_Midterm_Project\Lion_kings_RAG_Midterm_Project\documents",
    "chroma_db_path":  r"C:\1 Raji Sivani\UTD\Sem 4\Agentic AI\Project 1\Lion_kings_RAG_Midterm_Project\Lion_kings_RAG_Midterm_Project\documents",
    "collection_name": "national_parks1",

    # Chunking
    "chunk_size":      150,
    "chunk_overlap":   15,

    # Retrieval
    "top_k":           7,

    # Conversation
    "max_history":     6,
}

print("✅ Imports successful!")
print(f"   LLM Model    : {CONFIG['llm_model']}")
print(f"   Embed Model  : {CONFIG['embed_model']}")
print(f"   Documents    : {CONFIG['documents_dir']}")
print(f"   Database     : {CONFIG['chroma_db_path']}")

✅ Imports successful!
   LLM Model    : qwen2.5:1.5b
   Embed Model  : nomic-embed-text
   Documents    : C:\Users\14694\OneDrive - The University of Texas at Dallas\BUAN 6V99 Agentic AI\RAG\documents
   Database     : C:\Users\14694\OneDrive - The University of Texas at Dallas\BUAN 6V99 Agentic AI\RAG\chroma_db


2. Document Loader

In [4]:
def load_documents(directory: str) -> list:
    """
    Reads all .txt files from the given folder.
    Returns a list of dicts — one dict per document.
    """
    documents = []
    path = Path(directory)

    # Check if folder exists
    if not path.exists():
        print(f"❌ Folder not found: {directory}")
        return documents

    # Get all .txt files
    files = sorted(path.glob("*.txt"))

    if not files:
        print(f"⚠️  No .txt files found in: {directory}")
        return documents

    print(f"📂 Found {len(files)} files. Loading...")

    for file in files:
        try:
            content = file.read_text(encoding="utf-8").strip()

            if not content:
                print(f"   ⚠️  Skipped (empty): {file.name}")
                continue

            documents.append({
                "filename": file.name,
                "content":  content,
            })
            print(f"   ✅ {file.name}")

        except Exception as e:
            print(f"   ❌ Could not read {file.name}: {e}")

    print(f"\n📄 Total loaded: {len(documents)} documents")
    return documents

documents = load_documents(CONFIG["documents_dir"])

📂 Found 63 files. Loading...
   ✅ acadia.txt
   ✅ american_samoa.txt
   ✅ arches.txt
   ✅ badlands.txt
   ✅ big_bend.txt
   ✅ biscayne.txt
   ✅ black_canyon_gunnison.txt
   ✅ bryce_canyon.txt
   ✅ canyonlands.txt
   ✅ capitol_reef.txt
   ✅ carlsbad_caverns.txt
   ✅ channel_islands.txt
   ✅ congaree.txt
   ✅ crater_lake.txt
   ✅ cuyahoga_valley.txt
   ✅ death_valley.txt
   ✅ denali.txt
   ✅ dry_tortugas.txt
   ✅ everglades.txt
   ✅ gates_of_the_arctic.txt
   ✅ gateway_arch.txt
   ✅ glacier.txt
   ✅ glacier_bay.txt
   ✅ grand_canyon.txt
   ✅ grand_teton.txt
   ✅ great_basin.txt
   ✅ great_sand_dunes.txt
   ✅ great_smoky_mountains.txt
   ✅ guadalupe_mountains.txt
   ✅ haleakala.txt
   ✅ hawaii_volcanoes.txt
   ✅ hot_springs.txt
   ✅ indiana_dunes.txt
   ✅ isle_royale.txt
   ✅ joshua_tree.txt
   ✅ katmai.txt
   ✅ kenai_fjords.txt
   ✅ kings_canyon.txt
   ✅ kobuk_valley.txt
   ✅ lake_clark.txt
   ✅ lassen_volcanic.txt
   ✅ mammoth_cave.txt
   ✅ mesa_verde.txt
   ✅ mount_rainier.txt
   ✅ new

3. Text Chunker

In [5]:
# ── Cell 3: Text Chunker ──────────────────────────────────────────────

def chunk_document(filename: str, content: str, chunk_size: int, overlap: int) -> list:
    """
    Splits one document's text into overlapping word-based chunks.
    Returns a list of chunk dicts.
    """
    words  = content.split()
    chunks = []
    start  = 0
    index  = 0

    while start < len(words):
        # Grab a slice of words
        end        = min(start + chunk_size, len(words))
        chunk_text = " ".join(words[start:end])

        # Build a unique ID from filename + chunk index
        chunk_id = hashlib.md5(f"{filename}_{index}".encode()).hexdigest()[:12]

        chunks.append({
            "id":       chunk_id,
            "text":     chunk_text,
            "source":   filename,
            "index":    index,
        })

        index += 1
        start += chunk_size - overlap   # move forward but keep overlap words

    return chunks


def chunk_all_documents(documents: list) -> list:
    """
    Chunks every document and combines into one flat list.
    """
    all_chunks = []

    print(f"✂️  Chunking {len(documents)} documents...")
    print(f"   Chunk size: {CONFIG['chunk_size']} words | Overlap: {CONFIG['chunk_overlap']} words\n")

    for doc in documents:
        chunks = chunk_document(
            filename   = doc["filename"],
            content    = doc["content"],
            chunk_size = CONFIG["chunk_size"],
            overlap    = CONFIG["chunk_overlap"],
        )
        all_chunks.extend(chunks)
        print(f"   ✅ {doc['filename']:<45} → {len(chunks)} chunks")

    print(f"\n📊 Total chunks created: {len(all_chunks)}")
    return all_chunks


# ── Run it ────────────────────────────────────────────────────────────
all_chunks = chunk_all_documents(documents)

✂️  Chunking 63 documents...
   Chunk size: 150 words | Overlap: 15 words

   ✅ acadia.txt                                    → 8 chunks
   ✅ american_samoa.txt                            → 7 chunks
   ✅ arches.txt                                    → 8 chunks
   ✅ badlands.txt                                  → 8 chunks
   ✅ big_bend.txt                                  → 8 chunks
   ✅ biscayne.txt                                  → 7 chunks
   ✅ black_canyon_gunnison.txt                     → 8 chunks
   ✅ bryce_canyon.txt                              → 8 chunks
   ✅ canyonlands.txt                               → 8 chunks
   ✅ capitol_reef.txt                              → 8 chunks
   ✅ carlsbad_caverns.txt                          → 7 chunks
   ✅ channel_islands.txt                           → 8 chunks
   ✅ congaree.txt                                  → 7 chunks
   ✅ crater_lake.txt                               → 8 chunks
   ✅ cuyahoga_valley.txt                           → 7 ch

4. ChromaDB Indexer

In [6]:
# ── Cell 4: ChromaDB Indexer ──────────────────────────────────────────

def connect_chromadb() -> chromadb.Collection:
    """
    Connects to ChromaDB and returns the collection.
    Creates the chroma_db folder automatically if it doesn't exist.
    """
    client = chromadb.PersistentClient(
        path     = CONFIG["chroma_db_path"],
        settings = Settings(anonymized_telemetry=False),
    )

    collection = client.get_or_create_collection(
        name     = CONFIG["collection_name"],
        metadata = {"hnsw:space": "cosine"},
    )

    print(f"🗄️  ChromaDB connected!")
    print(f"   Collection : {CONFIG['collection_name']}")
    print(f"   Vectors stored so far: {collection.count()}")
    return collection


def embed_and_index(collection: chromadb.Collection, chunks: list) -> None:
    """
    Embeds each chunk using nomic-embed-text
    and stores the vectors in ChromaDB.
    Skips indexing if data already exists.
    """
    # Skip if already indexed
    if collection.count() > 0:
        print(f"✅ Already indexed! {collection.count()} vectors in database.")
        print("   Delete the chroma_db folder and re-run to rebuild.")
        return

    print(f"🔢 Embedding {len(chunks)} chunks...")
    print(f"   Model : {CONFIG['embed_model']}")
    print(f"   This runs ONCE and saves to disk.\n")

    start_time = time.time()

    for i, chunk in enumerate(chunks):
        try:
            # Step 1 — Convert text to vector
            result    = ollama.embeddings(
                model  = CONFIG["embed_model"],
                prompt = chunk["text"],
            )
            embedding = result["embedding"]

            # Step 2 — Store in ChromaDB
            collection.add(
                ids        = [chunk["id"]],
                documents  = [chunk["text"]],
                embeddings = [embedding],
                metadatas  = [{"source": chunk["source"],
                               "index":  chunk["index"]}],
            )

            # Progress update every 50 chunks
            if (i + 1) % 50 == 0 or (i + 1) == len(chunks):
                elapsed = round(time.time() - start_time, 1)
                pct     = round((i + 1) / len(chunks) * 100)
                print(f"   [{i+1}/{len(chunks)}] {pct}% done — {elapsed}s elapsed")

        except Exception as e:
            print(f"   ⚠️  Skipped chunk {i} ({chunk['source']}): {e}")
            continue

    total = round(time.time() - start_time, 1)
    print(f"\n✅ Indexing complete!")
    print(f"   Vectors stored : {collection.count()}")
    print(f"   Time taken     : {total}s")


# ── Run it ────────────────────────────────────────────────────────────
collection = connect_chromadb()
embed_and_index(collection, all_chunks)


🗄️  ChromaDB connected!
   Collection : national_parks1
   Vectors stored so far: 0
🔢 Embedding 514 chunks...
   Model : nomic-embed-text
   This runs ONCE and saves to disk.

   [50/514] 10% done — 56.2s elapsed
   [100/514] 19% done — 110.5s elapsed
   [150/514] 29% done — 168.7s elapsed
   [200/514] 39% done — 226.3s elapsed
   [250/514] 49% done — 277.5s elapsed
   [300/514] 58% done — 328.9s elapsed
   [350/514] 68% done — 379.9s elapsed
   [400/514] 78% done — 430.8s elapsed
   [450/514] 88% done — 484.7s elapsed
   [500/514] 97% done — 539.6s elapsed
   [514/514] 100% done — 556.6s elapsed

✅ Indexing complete!
   Vectors stored : 514
   Time taken     : 556.6s


In [7]:
print(f"Total documents loaded : {len(documents)}")
print(f"Total chunks created   : {len(all_chunks)}")
print()

# Show chunks per document
print("Chunks per document:")
print()

from collections import Counter
source_counts = Counter([chunk["source"] for chunk in all_chunks])

for source, count in sorted(source_counts.items()):
    print(f"   {source:<45} → {count} chunks")

Total documents loaded : 63
Total chunks created   : 514

Chunks per document:

   acadia.txt                                    → 8 chunks
   american_samoa.txt                            → 7 chunks
   arches.txt                                    → 8 chunks
   badlands.txt                                  → 8 chunks
   big_bend.txt                                  → 8 chunks
   biscayne.txt                                  → 7 chunks
   black_canyon_gunnison.txt                     → 8 chunks
   bryce_canyon.txt                              → 8 chunks
   canyonlands.txt                               → 8 chunks
   capitol_reef.txt                              → 8 chunks
   carlsbad_caverns.txt                          → 7 chunks
   channel_islands.txt                           → 8 chunks
   congaree.txt                                  → 7 chunks
   crater_lake.txt                               → 8 chunks
   cuyahoga_valley.txt                           → 7 chunks
   death_valley.txt 

5. Retriever

This cell's only job is to take a user's question, convert it to a vector, and find the most similar chunks in ChromaDB.

In [8]:
# ── Cell 5: Retriever ─────────────────────────────────────────────────

def retrieve(query: str, collection: chromadb.Collection) -> list:
    """
    Takes a question, embeds it, searches ChromaDB,
    and returns the top-k most relevant chunks.
    """

    # Step 1 — Embed the question using the SAME model
    result          = ollama.embeddings(
        model  = CONFIG["embed_model"],
        prompt = query,
    )
    query_embedding = result["embedding"]

    # Step 2 — Search ChromaDB for similar vectors
    results = collection.query(
        query_embeddings = [query_embedding],
        n_results        = CONFIG["top_k"],
        include          = ["documents", "metadatas", "distances"],
    )

    # Step 3 — Format and return results
    chunks = []
    for text, meta, distance in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0],
    ):
        chunks.append({
            "text":       text,
            "source":     meta["source"],
            "similarity": round(1 - distance, 4),
        })

    return chunks


# ── Test it ───────────────────────────────────────────────────────────
test_query   = "Which national park has the best wildlife watching?"
test_results = retrieve(test_query, collection)

print(f"🔍 Query: '{test_query}'\n")
for i, chunk in enumerate(test_results, 1):
    print(f"[{i}] Source     : {chunk['source']}")
    print(f"    Similarity : {chunk['similarity']}")
    print(f"    Preview    : {chunk['text'][:120]}...")
    print()



🔍 Query: 'Which national park has the best wildlife watching?'

[1] Source     : badlands.txt
    Similarity : 0.7792
    Preview    : - The Sage Creek area (western, less-visited part of the park) offers the best solitude and wildlife viewing....

[2] Source     : yellowstone.txt
    Similarity : 0.7761
    Preview    : best wildlife watching in the park. Wolf packs, grizzly bears, bison herds, elk, pronghorn, and coyotes are commonly spo...

[3] Source     : grand_teton.txt
    Similarity : 0.7624
    Preview    : of the most photographed spots in the park. 6. Signal Mountain Summit Road: A 5-mile paved road to the summit of Signal ...

[4] Source     : great_smoky_mountains.txt
    Similarity : 0.7571
    Preview    : one of the few places in North America where synchronous fireflies (Photinus carolinus) flash in unison. A lottery syste...

[5] Source     : wrangell_st_elias.txt
    Similarity : 0.753
    Preview    : park's interior. Dall sheep are frequently seen on rocky slopes a

6. Generator

In [ ]:
# ── Cell 6: Generator ─────────────────────────────────────────────────

SYSTEM_PROMPT = """You are a friendly and knowledgeable USA National Parks travel assistant.
You help visitors plan their trips by answering questions about national parks.

RULES:
1. Answer ONLY using the provided context.
2. If the user mentions a specific park by name, answer ONLY about that park.
   Ignore information from other parks in the context.
3. If the context doesn't contain the answer say "I don't have that information."
4. Always mention which park you are talking about.
5. Keep answers clear, helpful, and easy to read.
6. Include practical details like fees, trails, and tips when available.
7. Be concise — keep answers under 100 words."""


def generate(query: str, chunks: list, chat_history: list) -> tuple:
    """
    Builds a prompt from the retrieved chunks + chat history,
    sends it to gemma2:2b, and returns (answer, time_taken).
    """

    # Step 1 — Build context from retrieved chunks
    context_parts = []
    for i, chunk in enumerate(chunks, 1):
        context_parts.append(f"[Source {i}: {chunk['source']}]\n{chunk['text']}")
    context = "\n\n---\n\n".join(context_parts)

    # Step 2 — Build the full message list
    messages = []

    # System message — tells the LLM how to behave
    messages.append({
        "role":    "system",
        "content": SYSTEM_PROMPT,
    })

    # Chat history — gives the LLM memory of past exchanges
    messages.extend(chat_history)

    # Current question + context
    messages.append({
        "role": "user",
        "content": f"""Use the following context to answer the question.

CONTEXT:
{context}

QUESTION: {query}""",
    })

    # Step 3 — Send to LLM and measure time
    start  = time.time()
    response = ollama.chat(
        model    = CONFIG["llm_model"],
        messages = messages,
    )
    elapsed = round(time.time() - start, 2)

    answer = response["message"]["content"].strip()
    return answer, elapsed


# ── Test it ───────────────────────────────────────────────────────────
print("🤖 Generating answer...\n")

answer, gen_time = generate(
    query        = test_query,    # from Cell 5
    chunks       = test_results,  # from Cell 5
    chat_history = [],            # empty for now — Cell 7 handles this
)

print(f"Answer:\n{answer}")
print(f"\n⏱️  Generation time: {gen_time}s")

🤖 Generating answer...

Answer:
Yellowstone National Park has the best wildlife watching experience, as it offers the largest concentration of mammals in the lower 48 states, including bison, grizzly bears, black bears, gray wolves, elk, moose, pronghorn, bighorn sheep, mountain goats, coyotes, foxes, river otters, bald eagles, and more.

⏱️  Generation time: 103.59s


7.Conversation Memory

In [11]:
# ── Cell 7: Conversation Memory ───────────────────────────────────────

# Simple list that stores the conversation
chat_history = []

def add_to_memory(role: str, content: str) -> None:
    """
    Adds a message to chat history.
    Trims to max_history when it gets too long.

    role → either "user" or "assistant"
    """
    chat_history.append({
        "role":    role,
        "content": content,
    })

    # Keep only the most recent messages
    max_msgs = CONFIG["max_history"]
    if len(chat_history) > max_msgs:
        del chat_history[:-max_msgs]


def clear_memory() -> None:
    """Wipes the conversation history completely."""
    chat_history.clear()
    print("🔄 Memory cleared!")


def show_memory() -> None:
    """Prints the current conversation history."""
    if not chat_history:
        print("   (Memory is empty)")
        return

    print(f"📝 Chat History ({len(chat_history)} messages):\n")
    for i, msg in enumerate(chat_history, 1):
        role    = "🧑 User" if msg["role"] == "user" else "🤖 Assistant"
        preview = msg["content"][:80].replace("\n", " ")
        print(f"   [{i}] {role}: {preview}...")



8. RAG Pipeline

In [12]:
# ── Cell 8: RAG Pipeline ──────────────────────────────────────────────

def ask(query: str) -> dict:
    """
    The complete RAG pipeline in one function.

    Takes a question → retrieves chunks → generates answer
    → saves to memory → returns everything.
    """

    # ── Step 1: Retrieve relevant chunks ──────────────────────────────
    t0     = time.time()
    chunks = retrieve(query, collection)
    t1     = round(time.time() - t0, 2)

    # ── Step 2: Generate answer ────────────────────────────────────────
    answer, gen_time = generate(query, chunks, chat_history)

    # ── Step 3: Save to memory ─────────────────────────────────────────
    add_to_memory("user",      query)
    add_to_memory("assistant", answer)

    # ── Step 4: Return everything ──────────────────────────────────────
    return {
        "query":           query,
        "answer":          answer,
        "sources":         list(dict.fromkeys([c["source"] for c in chunks])),
        "chunks":          chunks,
        "retrieval_time":  t1,
        "generation_time": gen_time,
        "total_time":      round(t1 + gen_time, 2),
    }




9.StreamlitUI

In [20]:
# ── Cell 9: Streamlit UI ──────────────────────────────────────────────

app_code = '''
import streamlit as st
import time
import hashlib
import ollama
import chromadb
from pathlib import Path
from chromadb.config import Settings

# ── CONFIG ────────────────────────────────────────────────────────────
CONFIG = {
    "llm_model":       "gemma2:2b",
    "embed_model":     "nomic-embed-text",
    "documents_dir":   r"C:\\Users\\14694\\OneDrive - The University of Texas at Dallas\\BUAN 6V99 Agentic AI\\RAG\\documents",
    "chroma_db_path":  r"C:\\Users\\14694\\OneDrive - The University of Texas at Dallas\\BUAN 6V99 Agentic AI\\RAG\\chroma_db",
    "collection_name": "national_parks1",
    "chunk_size":      150,
    "chunk_overlap":   15,
    "top_k":           7,
    "max_history":     6,
    "num_predict":     80,    # ← max words in answer (speeds up generation)
}

SYSTEM_PROMPT = """You are a friendly and knowledgeable USA National Parks travel assistant.
You help visitors plan their trips by answering questions about national parks.

RULES:
1. Answer ONLY using the provided context.
2. If the user mentions a specific park by name, answer ONLY about that park.
   Ignore information from other parks in the context.
3. If the context doesn't contain the answer say "I don't have that information."
4. Always mention which park you are talking about.
5. Keep answers clear, helpful, and easy to read.
6. Include practical details like fees, trails, and tips when available.
7. Be concise — keep answers under 100 words."""

# ── Load ChromaDB once ────────────────────────────────────────────────
@st.cache_resource
def load_collection():
    client = chromadb.PersistentClient(
        path     = CONFIG["chroma_db_path"],
        settings = Settings(anonymized_telemetry=False),
    )
    collection = client.get_or_create_collection(
        name     = CONFIG["collection_name"],
        metadata = {"hnsw:space": "cosine"},
    )
    return collection

# ── Enrich query with memory ──────────────────────────────────────────
def enrich_query(query, chat_history):
    """
    Combines current question with recent chat history
    so ChromaDB finds the RIGHT document even for
    follow-up questions like "what is the entry fee?"
    """
    if not chat_history:
        return query

    # Take last 2 messages for context
    recent       = chat_history[-4:]
    history_text = " ".join([m["content"] for m in recent])

    # Combine for a richer search query
    enriched = f"{history_text} {query}"
    return enriched

# ── Retrieve ──────────────────────────────────────────────────────────
def retrieve(query, collection):
    result = ollama.embeddings(
        model  = CONFIG["embed_model"],
        prompt = query,
    )
    query_embedding = result["embedding"]

    results = collection.query(
        query_embeddings = [query_embedding],
        n_results        = CONFIG["top_k"],
        include          = ["documents", "metadatas", "distances"],
    )

    chunks = []
    for text, meta, distance in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0],
    ):
        chunks.append({
            "text":       text,
            "source":     meta["source"],
            "similarity": round(1 - distance, 4),
        })
    return chunks

# ── Generate ──────────────────────────────────────────────────────────
def generate(query, chunks, chat_history):
    context_parts = []
    for i, chunk in enumerate(chunks, 1):
        context_parts.append(
            f"[Source {i}: {chunk[\'source\']}]\\n{chunk[\'text\']}"
        )
    context = "\\n\\n---\\n\\n".join(context_parts)

    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    messages.extend(chat_history)
    messages.append({
        "role":    "user",
        "content": f"""Use the following context to answer the question.

CONTEXT:
{context}

QUESTION: {query}""",
    })

    start    = time.time()
    response = ollama.chat(
        model    = CONFIG["llm_model"],
        messages = messages,
        options  = {
            "num_predict": 100,  # ← limits response length
            "temperature": 0.7,
            "num_ctx":     2048,   # ← limits context window size
            "top_k":       10,     # ← faster token selection
            "top_p":       0.9,

        }
    )
    elapsed = round(time.time() - start, 2)
    answer  = response["message"]["content"].strip()
    return answer, elapsed

# ── Page config ───────────────────────────────────────────────────────
st.set_page_config(
    page_title = "National Parks Assistant",
    page_icon  = "🏞️",
    layout     = "centered",
)

st.title("🏞️ USA National Parks Assistant")
st.caption("Ask me anything about the 63 US National Parks!")
st.divider()

# ── Load collection ───────────────────────────────────────────────────
collection = load_collection()

# ── Session state ─────────────────────────────────────────────────────
if "messages" not in st.session_state:
    st.session_state.messages      = []

if "chat_history" not in st.session_state:
    st.session_state.chat_history  = []

# ── Display existing messages ─────────────────────────────────────────
for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])
        

# ── Chat input ────────────────────────────────────────────────────────
if prompt := st.chat_input("Ask about any national park..."):

    # Show user message immediately
    with st.chat_message("user"):
        st.markdown(prompt)

    st.session_state.messages.append({
        "role":    "user",
        "content": prompt,
    })

    # Generate response
    with st.chat_message("assistant"):
        with st.spinner("Thinking... (may take 1-2 minutes on CPU)"):

            # Enrich query with memory before retrieving
            enriched_query  = enrich_query(
                prompt,
                st.session_state.chat_history
            )

            # Retrieve using enriched query
            t0              = time.time()
            chunks          = retrieve(enriched_query, collection)
            retrieval_time  = round(time.time() - t0, 2)

            # Generate using original prompt (not enriched)
            answer, gen_time = generate(
                prompt,
                chunks,
                st.session_state.chat_history,
            )

            sources = list(dict.fromkeys([c["source"] for c in chunks]))
            timing  = f"Retrieved in {retrieval_time}s | Generated in {gen_time}s"

        st.markdown(answer)
       

    # Save to display history
    st.session_state.messages.append({
        "role":    "assistant",
        "content": answer,
        "sources": sources,
        "timing":  timing,
    })

    # Update conversation memory
    st.session_state.chat_history.append({"role": "user",      "content": prompt})
    st.session_state.chat_history.append({"role": "assistant",  "content": answer})

    # Trim memory to max_history
    max_msgs = CONFIG["max_history"]
    if len(st.session_state.chat_history) > max_msgs:
        st.session_state.chat_history = st.session_state.chat_history[-max_msgs:]

# ── Clear chat button ─────────────────────────────────────────────────
if st.session_state.messages:
    if st.button("🔄 Clear Chat"):
        st.session_state.messages     = []
        st.session_state.chat_history = []
        st.rerun()
'''

# ── Write app.py to disk ──────────────────────────────────────────────
app_path = r"C:\Users\14694\OneDrive - The University of Texas at Dallas\BUAN 6V99 Agentic AI\RAG\app.py"

with open(app_path, "w", encoding="utf-8") as f:
    f.write(app_code)

print("✅ app.py updated successfully!")
print()
print("Two fixes applied:")
print("  1. num_predict: 150  → answers limited to ~150 words (~2 min)")
print("  2. enrich_query()    → follow-up questions now find correct park")
print()
print("🚀 Launch with:")
print(f'   python -m streamlit run "{app_path}"')


✅ app.py updated successfully!

Two fixes applied:
  1. num_predict: 150  → answers limited to ~150 words (~2 min)
  2. enrich_query()    → follow-up questions now find correct park

🚀 Launch with:
   python -m streamlit run "C:\Users\14694\OneDrive - The University of Texas at Dallas\BUAN 6V99 Agentic AI\RAG\app.py"


In [15]:

# ── Debug Cell ────────────────────────────────────────────
query  = "where is yellowstone national park"
chunks = retrieve(query, collection)

print(f"Query: '{query}'")
print(f"Top {CONFIG['top_k']} chunks retrieved:\n")

for i, chunk in enumerate(chunks, 1):
    print(f"[{i}] Source     : {chunk['source']}")
    print(f"     Similarity : {chunk['similarity']}")
    print(f"     Preview    : {chunk['text'][:100]}...")
    print()


Query: 'where is yellowstone national park'
Top 3 chunks retrieved:

[1] Source     : katmai.txt
     Similarity : 0.7252
     Preview    : Katmai National Park - Complete Guide Overview: Katmai National Park and Preserve is located on the ...

[2] Source     : pinnacles.txt
     Similarity : 0.7128
     Preview    : Pinnacles National Park - Complete Guide Overview: Pinnacles National Park is located in central Cal...

[3] Source     : hawaii_volcanoes.txt
     Similarity : 0.7106
     Preview    : Hawaii Volcanoes National Park - Complete Guide Overview: Hawaii Volcanoes National Park is located ...



In [26]:
# ── Debug Cell: Check if Yellowstone is in ChromaDB ───────────────────

# Check total vectors
print(f"Total vectors in ChromaDB: {collection.count()}")
print()

# Search specifically for yellowstone
results = collection.get(
    where = {"source": {"$eq": "yellowstone.txt"}}
)

print(f"Yellowstone chunks found: {len(results['ids'])}")
print()

if results['ids']:
    print("✅ Yellowstone IS in the database")
    print(f"   Number of chunks: {len(results['ids'])}")
    print(f"   Preview of chunk 1: {results['documents'][0][:150]}...")
else:
    print("❌ Yellowstone is NOT in the database!")
    print("   This is why it never gets retrieved.")


Total vectors in ChromaDB: 193

Yellowstone chunks found: 2

✅ Yellowstone IS in the database
   Number of chunks: 2
   Preview of chunk 1: Yellowstone National Park - Complete Guide Overview: Yellowstone is the first and oldest national park in the world, established on March 1, 1872. Loc...


In [14]:
# Check indexing status
print(f"Total vectors: {collection.count()}")
print()

# Check Acadia specifically
results = collection.get(
    where={"source": {"$eq": "acadia.txt"}}
)
print(f"Acadia chunks found: {len(results['ids'])}")

# Check how many unique parks are indexed
import pandas as pd
all_data = collection.get(include=["metadatas"])
sources  = [m["source"] for m in all_data["metadatas"]]
unique   = sorted(set(sources))
print(f"\nUnique parks indexed: {len(unique)}")
print()
for s in unique:
    print(f"  {s}")

Total vectors: 514

Acadia chunks found: 8

Unique parks indexed: 63

  acadia.txt
  american_samoa.txt
  arches.txt
  badlands.txt
  big_bend.txt
  biscayne.txt
  black_canyon_gunnison.txt
  bryce_canyon.txt
  canyonlands.txt
  capitol_reef.txt
  carlsbad_caverns.txt
  channel_islands.txt
  congaree.txt
  crater_lake.txt
  cuyahoga_valley.txt
  death_valley.txt
  denali.txt
  dry_tortugas.txt
  everglades.txt
  gates_of_the_arctic.txt
  gateway_arch.txt
  glacier.txt
  glacier_bay.txt
  grand_canyon.txt
  grand_teton.txt
  great_basin.txt
  great_sand_dunes.txt
  great_smoky_mountains.txt
  guadalupe_mountains.txt
  haleakala.txt
  hawaii_volcanoes.txt
  hot_springs.txt
  indiana_dunes.txt
  isle_royale.txt
  joshua_tree.txt
  katmai.txt
  kenai_fjords.txt
  kings_canyon.txt
  kobuk_valley.txt
  lake_clark.txt
  lassen_volcanic.txt
  mammoth_cave.txt
  mesa_verde.txt
  mount_rainier.txt
  new_river_gorge.txt
  north_cascades.txt
  olympic.txt
  petrified_forest.txt
  pinnacles.txt
  r

In [15]:
# Debug Cell — test Acadia retrieval
query  = "where is acadia national park located"
chunks = retrieve(query, collection)

print(f"Query: '{query}'\n")
for i, chunk in enumerate(chunks, 1):
    print(f"[{i}] Source     : {chunk['source']}")
    print(f"     Similarity : {chunk['similarity']}")
    print(f"     Preview    : {chunk['text'][:120]}...")
    print()


Query: 'where is acadia national park located'

[1] Source     : acadia.txt
     Similarity : 0.6931
     Preview    : Acadia National Park - Complete Guide Overview: Acadia National Park is located primarily on Mount Desert Island on the ...

[2] Source     : indiana_dunes.txt
     Similarity : 0.6883
     Preview    : Indiana Dunes National Park - Complete Guide Overview: Indiana Dunes National Park is located along the southern shore o...

[3] Source     : glacier_bay.txt
     Similarity : 0.6796
     Preview    : Glacier Bay National Park - Complete Guide Overview: Glacier Bay National Park and Preserve is located in southeastern A...

[4] Source     : cuyahoga_valley.txt
     Similarity : 0.6751
     Preview    : Cuyahoga Valley National Park - Complete Guide Overview: Cuyahoga Valley National Park is located in northeastern Ohio b...

[5] Source     : olympic.txt
     Similarity : 0.6719
     Preview    : Olympic National Park - Complete Guide Overview: Olympic National Park is l

In [19]:
query  = "what campgrounds are available in olympic national park"
chunks = retrieve(query, collection)

print(f"Query: '{query}'\n")
for i, chunk in enumerate(chunks, 1):
    print(f"[{i}] Source     : {chunk['source']}")
    print(f"     Similarity : {chunk['similarity']}")
    print(f"     Preview    : {chunk['text'][:200]}...")
    print()
    

Query: 'what campgrounds are available in olympic national park'

[1] Source     : hot_springs.txt
     Similarity : 0.7526
     Preview    : is no campground in the national park itself. Gulley Park and Lake Catherine State Park nearby provide camping options. The city of Hot Springs and Garvan Woodland Gardens area have private RV campgro...

[2] Source     : sequoia.txt
     Similarity : 0.7363
     Preview    : a quota reservation system via recreation.gov. Camping: Lodgepole Campground (214 sites) is the largest and most central campground, located near the Lodgepole Visitor Center and accessible year-round...

[3] Source     : arches.txt
     Similarity : 0.732
     Preview    : miles round trip, easy. 8. Devils Garden Primitive Loop — 7.9 miles round trip, strenuous, visits 8 arches. Camping: Devils Garden Campground (51 sites) is the only campground in the park, located at ...

[4] Source     : bryce_canyon.txt
     Similarity : 0.7203
     Preview    : Camping: The park has tw